## Project Description
This project works with a real-world supply chain dataset from Just In Time, a distribution company. Acting as the company's data analyst, I set out to solve key shipment and inventory management challenges, analyze supply chain inefficiencies, and build dashboards that inform business stakeholders about potential problems and support structural business improvements.

## Objective
In this project, my primary focus is on addressing key challenges related to shipment and inventory management within the supply chain. To achieve this goal efficiently, the project has been divided into few objectives:

# Data Preparation

In [1]:
# import the libraries
import pandas as pd
import numpy as np

In [2]:
# import datasets
from pathlib import Path

DATA = Path('Datasets')

df_orders      = pd.read_csv(DATA / 'Orders_and_shipments.csv', encoding='ISO-8859-1')
df_inventory   = pd.read_csv(DATA / 'Inventory.csv')
df_fulfillment = pd.read_csv(DATA / 'Fulfillment.csv')

# Several column names ship with leading/trailing spaces (' Order ID ').
# Strip them at load so every reference below is clean.
for df in (df_orders, df_inventory, df_fulfillment):
    df.columns = df.columns.str.strip()

print(f"orders      : {df_orders.shape}")
print(f"inventory   : {df_inventory.shape}")
print(f"fulfillment : {df_fulfillment.shape}")

orders      : (30871, 24)
inventory   : (4200, 4)
fulfillment : (118, 2)


In [3]:
df_orders.head()

,Order ID,Order Item ID,Order YearMonth,Order Year,Order Month,Order Day,Order Time,Order Quantity,Product Department,Product Category,...,Customer Country,Warehouse Country,Shipment Year,Shipment Month,Shipment Day,Shipment Mode,Shipment Days - Scheduled,Gross Sales,Discount %,Profit
0,3535,8793,201502,2015,2,21,14:07,1,Fan Shop,Fishing,...,Mexico,Puerto Rico,2015,2,27,Standard Class,4,400,0.25,200
1,4133,10320,201503,2015,3,2,07:37,1,Fan Shop,Fishing,...,Brazil,Puerto Rico,2015,3,6,Standard Class,4,400,0.09,200
2,7396,18517,201504,2015,4,18,22:47,1,Fan Shop,Fishing,...,Mexico,Puerto Rico,2015,4,20,Standard Class,4,400,0.06,200
3,11026,27608,201506,2015,6,10,22:32,1,Fan Shop,Fishing,...,Denmark,Puerto Rico,2015,6,12,Standard Class,4,400,0.15,200
4,11026,27609,201506,2015,6,10,22:32,1,Fan Shop,Fishing,...,Denmark,Puerto Rico,2015,6,12,Standard Class,4,400,0.13,200


In [4]:
df_inventory.head()

,Product Name,Year Month,Warehouse Inventory,Inventory Cost Per Unit
0,Perfect Fitness Perfect Rip Deck,201712,0,0.69517
1,Nike Men's Dri-FIT Victory Golf Polo,201712,2,1.29291
2,O'Brien Men's Neoprene Life Vest,201712,0,0.56531
3,Nike Men's Free 5.0+ Running Shoe,201712,1,1.26321
4,Under Armour Girls' Toddler Spine Surge Runni,201712,0,1.47648


In [5]:
df_fulfillment.head()

,Product Name,Warehouse Order Fulfillment (days)
0,Perfect Fitness Perfect Rip Deck,8.3
1,Nike Men's Dri-FIT Victory Golf Polo,6.6
2,O'Brien Men's Neoprene Life Vest,5.5
3,Nike Men's Free 5.0+ Running Shoe,9.4
4,Under Armour Girls' Toddler Spine Surge Runni,6.3


The dataset provides three data tables including order_and_shipment, inventory and fulfillment. After examining the data fields, I noticed that the dataset generally represents the following key information

* Customer: General information about customers including identifiers and addresses

* Order: Information about the order including date of order, product and quantity ordered, order value

* Shipment: Shipping information including shipping date, shipping mode

* Product: Specific information about the ordered item including product name, product category, product department

* Warehouse Inventory: Information on inventory management for each product name including monthly inventory, warehouse location, storage costs, order fulfillment

# Data Cleaning

## Handling Missing Value

In [6]:
df_orders.isna().sum()

Order ID                     0
Order Item ID                0
Order YearMonth              0
Order Year                   0
Order Month                  0
Order Day                    0
Order Time                   0
Order Quantity               0
Product Department           0
Product Category             0
Product Name                 0
Customer ID                  0
Customer Market              0
Customer Region              0
Customer Country             0
Warehouse Country            0
Shipment Year                0
Shipment Month               0
Shipment Day                 0
Shipment Mode                0
Shipment Days - Scheduled    0
Gross Sales                  0
Discount %                   0
Profit                       0
dtype: int64

In [7]:
df_inventory.isna().sum()

Product Name               0
Year Month                 0
Warehouse Inventory        0
Inventory Cost Per Unit    0
dtype: int64

In [8]:
df_fulfillment.isna().sum()

Product Name                          0
Warehouse Order Fulfillment (days)    0
dtype: int64

`isna()` reports no nulls in any of the three tables. That is not the same as
having no missing data.

The `Discount %` column stores missing values as the literal string `'  -  '`,
which `isna()` cannot see because it is a non-empty string. It is handled
explicitly during wrangling below rather than being silently treated as a real
zero discount.

## Duplicated Data

In [9]:
# `Order Item ID` is unique by construction, so a full-row duplicated() check
# cannot return anything but 0 - it proves nothing about data quality.
# Test a business key instead: the same customer buying the same product at the
# same moment would be a genuine double-entry.
print("full-row duplicates        :", df_orders.duplicated().sum())
print("duplicate Order Item IDs   :", df_orders.duplicated(subset=['Order Item ID']).sum())
print("same customer+product+time :", df_orders.duplicated(
    subset=['Customer ID', 'Product Name',
            'Order Year', 'Order Month', 'Order Day', 'Order Time']).sum())

full-row duplicates        : 0
duplicate Order Item IDs   : 0
same customer+product+time : 2128


In [10]:
# Inventory should hold one row per product per month.
print("full-row duplicates             :", df_inventory.duplicated().sum())
print("duplicate (Product, Year Month) :",
      df_inventory.duplicated(subset=['Product Name', 'Year Month']).sum())

full-row duplicates             : 0
duplicate (Product, Year Month) : 0


In [11]:
# Fulfillment should hold one row per product.
print("full-row duplicates     :", df_fulfillment.duplicated().sum())
print("duplicate Product Names :", df_fulfillment.duplicated(subset=['Product Name']).sum())

full-row duplicates     : 0
duplicate Product Names : 0


No duplicates surface on any of the business keys either, so the tables are
genuinely free of double-entry - a stronger statement than the full-row check
alone could support.

# Data Wrangling

## Leading and Trailing Space

In [12]:
df_orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 30871 entries, 0 to 30870
Data columns (total 24 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Order ID                   30871 non-null  int64
 1   Order Item ID              30871 non-null  int64
 2   Order YearMonth            30871 non-null  int64
 3   Order Year                 30871 non-null  int64
 4   Order Month                30871 non-null  int64
 5   Order Day                  30871 non-null  int64
 6   Order Time                 30871 non-null  str  
 7   Order Quantity             30871 non-null  int64
 8   Product Department         30871 non-null  str  
 9   Product Category           30871 non-null  str  
 10  Product Name               30871 non-null  str  
 11  Customer ID                30871 non-null  int64
 12  Customer Market            30871 non-null  str  
 13  Customer Region            30871 non-null  str  
 14  Customer Country           30871 

In [13]:
df_inventory.info()

<class 'pandas.DataFrame'>
RangeIndex: 4200 entries, 0 to 4199
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Product Name             4200 non-null   str    
 1   Year Month               4200 non-null   int64  
 2   Warehouse Inventory      4200 non-null   int64  
 3   Inventory Cost Per Unit  4200 non-null   float64
dtypes: float64(1), int64(2), str(1)
memory usage: 131.4 KB


In [14]:
df_fulfillment.info()

<class 'pandas.DataFrame'>
RangeIndex: 118 entries, 0 to 117
Data columns (total 2 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Product Name                        118 non-null    str    
 1   Warehouse Order Fulfillment (days)  118 non-null    float64
dtypes: float64(1), str(1)
memory usage: 2.0 KB


Upon inspecting the data using `.info()`, it has come to our attention that there are anomalies present in the datasets. Specifically, we have discovered that some columns contain leading or trailing spaces, which are not intended to be part of the column names.

In [15]:
# Column names were stripped at load time. Confirm none slipped through.
for name, df in [('orders', df_orders), ('inventory', df_inventory),
                 ('fulfillment', df_fulfillment)]:
    stray = [c for c in df.columns if c != c.strip()]
    print(f"{name:12s}: {len(stray)} columns with stray whitespace")

orders      : 0 columns with stray whitespace
inventory   : 0 columns with stray whitespace
fulfillment : 0 columns with stray whitespace


In [16]:
df_orders.columns

Index(['Order ID', 'Order Item ID', 'Order YearMonth', 'Order Year',
       'Order Month', 'Order Day', 'Order Time', 'Order Quantity',
       'Product Department', 'Product Category', 'Product Name', 'Customer ID',
       'Customer Market', 'Customer Region', 'Customer Country',
       'Warehouse Country', 'Shipment Year', 'Shipment Month', 'Shipment Day',
       'Shipment Mode', 'Shipment Days - Scheduled', 'Gross Sales',
       'Discount %', 'Profit'],
      dtype='str')

In [17]:
df_inventory.columns

Index(['Product Name', 'Year Month', 'Warehouse Inventory',
       'Inventory Cost Per Unit'],
      dtype='str')

In [18]:
df_fulfillment.columns

Index(['Product Name', 'Warehouse Order Fulfillment (days)'], dtype='str')

In [19]:
df_orders.columns

Index(['Order ID', 'Order Item ID', 'Order YearMonth', 'Order Year',
       'Order Month', 'Order Day', 'Order Time', 'Order Quantity',
       'Product Department', 'Product Category', 'Product Name', 'Customer ID',
       'Customer Market', 'Customer Region', 'Customer Country',
       'Warehouse Country', 'Shipment Year', 'Shipment Month', 'Shipment Day',
       'Shipment Mode', 'Shipment Days - Scheduled', 'Gross Sales',
       'Discount %', 'Profit'],
      dtype='str')

In [20]:
df_orders['Discount %'].sample(30)

29275     0.13
6496      0.01
3265      0.18
5207       -  
5075      0.07
27977      0.2
8670      0.05
29260     0.16
27445      0.2
30088     0.03
12581     0.15
17081     0.05
20254     0.02
4416      0.03
22485     0.06
4992      0.25
4281      0.16
29899     0.06
17630     0.13
6975       -  
8786      0.04
26013     0.02
1673      0.12
9506      0.01
20533     0.18
16482     0.18
6044      0.15
4682       -  
1171      0.03
19579     0.09
Name: Discount %, dtype: str

A sample of `Discount %` shows rows holding the string `'  -  '` rather than a
number, which is why the column loaded as `object` instead of `float`.

These are **missing** discounts. Coercing them to `0` would assert that those
orders carried no discount at all - a factual claim the data does not make.
They are converted to `NaN` so that aggregates exclude them rather than being
dragged toward zero.

In [21]:
# '  -  ' marks a *missing* discount, not a zero discount. Mapping it to 0 would
# state that ~1,700 orders were sold at full price, pulling the mean discount
# down and misstating any discount analysis built on top.
# NaN is the honest encoding - Tableau excludes nulls from aggregates.
missing_discount = (df_orders['Discount %'] == '  -  ').sum()

df_orders['Discount %'] = pd.to_numeric(
    df_orders['Discount %'].replace('  -  ', np.nan), errors='coerce')

print(f"discounts recorded as '  -  ' -> NaN : {missing_discount:,}")
print(f"share of all rows                    : {missing_discount / len(df_orders) * 100:.1f}%")
print(f"mean discount, nulls excluded        : {df_orders['Discount %'].mean():.4f}")

discounts recorded as '  -  ' -> NaN : 1,749
share of all rows                    : 5.7%
mean discount, nulls excluded        : 0.1074


In [22]:
df_orders.columns

Index(['Order ID', 'Order Item ID', 'Order YearMonth', 'Order Year',
       'Order Month', 'Order Day', 'Order Time', 'Order Quantity',
       'Product Department', 'Product Category', 'Product Name', 'Customer ID',
       'Customer Market', 'Customer Region', 'Customer Country',
       'Warehouse Country', 'Shipment Year', 'Shipment Month', 'Shipment Day',
       'Shipment Mode', 'Shipment Days - Scheduled', 'Gross Sales',
       'Discount %', 'Profit'],
      dtype='str')

## Feature Engineering :
Order Datetime & Shipment Datetime

Due to the data being separated into multiple columns for year, month, day, and time, resulting in a large number of columns, I have decided to combine these columns into a single column that includes all the necessary date and time information. This will help simplify the data and make it more manageable for further analysis in Tableau.

In [23]:
# Make new columns: Order Datetime and Shipment Datetime
df_orders['Order Datetime'] = pd.to_datetime(df_orders['Order Year'].astype(str) + '-' + df_orders['Order Month'].astype(str) + '-' + df_orders['Order Day'].astype(str) + ' ' + df_orders['Order Time'])
df_orders['Shipment Datetime'] = pd.to_datetime(df_orders['Shipment Year'].astype(str) + '-' + df_orders['Shipment Month'].astype(str) + '-' + df_orders['Shipment Day'].astype(str))

# Displaying the result
df_orders[['Order Datetime', 'Shipment Datetime']].head()

,Order Datetime,Shipment Datetime
0,2015-02-21 14:07:00,2015-02-27
1,2015-03-02 07:37:00,2015-03-06
2,2015-04-18 22:47:00,2015-04-20
3,2015-06-10 22:32:00,2015-06-12
4,2015-06-10 22:32:00,2015-06-12


In [24]:
# Drop unnecessary columns
df_orders.drop(columns=['Order Year', 'Order Month', 'Order Day', 'Order Time',
                        'Shipment Year', 'Shipment Month', 'Shipment Day'], inplace=True)

### Remving unwanted characters in country name

In [25]:
df_orders['Customer Country'].unique()

<StringArray>
[       'Mexico',        'Brazil',       'Denmark',   'Netherlands',
       'Germany',         'China',     'Indonesia',      'Pakistan',
         'India',           'USA',
 ...
         'Qatar',  'Sierra Leona',      'Slovakia',    'Martinique',
        'Uganda',       'Namibia',      'Paraguay',          'Oman',
 'French Guiana',         'Nepal']
Length: 139, dtype: str

In [26]:
#replace the special characters in the Customer Country column
df_orders['Customer Country'] = df_orders['Customer Country'].replace({
    'Dominican\xa0Republic': 'Dominican Republic',
    'Cote d\x92Ivoire': 'Cote d Ivoire', # Added a comma at the end of this line
    'Perú': 'Peru',
    'Algeria\xa0': 'Algeria',
    'Israel\xa0':'Israel',
    'Benín': 'Benin'
})
df_orders['Customer Country'].unique()

<StringArray>
[       'Mexico',        'Brazil',       'Denmark',   'Netherlands',
       'Germany',         'China',     'Indonesia',      'Pakistan',
         'India',           'USA',
 ...
         'Qatar',  'Sierra Leona',      'Slovakia',    'Martinique',
        'Uganda',       'Namibia',      'Paraguay',          'Oman',
 'French Guiana',         'Nepal']
Length: 139, dtype: str

# Data Manipulation

## Order Processing Time

The "`Order Processing Time`" is obtained by subtracting the "`Order Datetime`" from the "`Shipment Datetime`".

It measures the time taken for the order to move through various stages, including **processing, packing, and delivery preparation**, until it is ready for shipment.

This metric helps us understand how quickly we can fulfill customer orders and deliver products to their destination.

In [27]:
# `Order Datetime` carries a clock time; `Shipment Datetime` is midnight. A raw
# subtraction therefore truncates downward and loses a day on almost every row -
# 29,975 of 30,871. Normalising the order timestamp to midnight first compares
# calendar dates, which is what "processing time in days" actually means.
df_orders['Order Processing Time'] = (
    df_orders['Shipment Datetime'] - df_orders['Order Datetime'].dt.normalize()
).dt.days

df_orders['Order Processing Time'].describe(percentiles=[.25, .5, .75, .95]).round(2)

count    30871.00
mean         3.56
std        131.20
min       -975.00
25%          2.00
50%          3.00
75%          5.00
95%        116.00
max        978.00
Name: Order Processing Time, dtype: float64

### Validating Shipment Dates

Correcting the off-by-one exposes a larger problem the original calculation hid.

A shipment cannot precede its own order, yet **2,735 rows** carry a shipment date
*before* the order date - by as much as 975 days - and a further **1,683 rows**
show processing times beyond 90 days. Together roughly **16% of rows carry
unusable shipment dates**.

These rows are **flagged, not deleted**. Their order values, quantities and
product details are all valid, so removing them would distort revenue and volume
figures. Instead a `Valid Processing Time` flag lets every downstream
shipment-timing metric exclude them explicitly, while value-based analysis keeps
the full dataset.

In [28]:
PLAUSIBLE_MAX_DAYS = 30

df_orders['Valid Processing Time'] = df_orders['Order Processing Time'].between(0, PLAUSIBLE_MAX_DAYS)

n_negative = (df_orders['Order Processing Time'] < 0).sum()
n_extreme  = (df_orders['Order Processing Time'] > PLAUSIBLE_MAX_DAYS).sum()
n_flagged  = (~df_orders['Valid Processing Time']).sum()
valid      = df_orders.loc[df_orders['Valid Processing Time'], 'Order Processing Time']

print(f"shipped before it was ordered : {n_negative:>6,}  ({n_negative / len(df_orders) * 100:.1f}%)")
print(f"processing time > {PLAUSIBLE_MAX_DAYS} days       : {n_extreme:>6,}  ({n_extreme / len(df_orders) * 100:.1f}%)")
print(f"total flagged as unusable     : {n_flagged:>6,}  ({n_flagged / len(df_orders) * 100:.1f}%)")
print(f"\nmost extreme negative value   : {df_orders['Order Processing Time'].min():,.0f} days")
print(f"median processing time (valid): {valid.median():.0f} days")
print(f"mean processing time   (valid): {valid.mean():.2f} days")

shipped before it was ordered :  2,735  (8.9%)
processing time > 30 days       :  2,254  (7.3%)
total flagged as unusable     :  4,989  (16.2%)

most extreme negative value   : -975 days
median processing time (valid): 3 days
mean processing time   (valid): 3.50 days


`Order Datetime` includes a clock time, but `Shipment Datetime` does not - the
source data carries no shipment time, so it defaults to `00:00:00`.

Subtracting the two directly and taking `.dt.days` truncates toward zero, which
silently removes a day from **29,975 of 30,871 rows** - any order not placed at
exactly midnight. The earlier approach patched only the same-day case
(`-1 -> 0`) and left the systematic error in place across the rest of the file.

Normalising the order timestamp to midnight before subtracting compares calendar
dates directly and removes the bias entirely.

## Feature Metrics
The feature metrics that planning to create in tableau which helps to further analyze inventory management, Shipement delay and many more.

### Total Inventory Storage Cost

Total Cost of Inventory is a significant financial metric in supply chain management that calculates the overall cost associated with holding and managing inventory.

The formula for calculating Total Inventory Cost is: `Warehouse Inventory` times `Inventory Cost per Unit`.

#### Storage Cost
Storage_Cost = [Inventory Cost Per Unit]*[Warehouse Inventory]

### Shipment Delay

Shipment Delay is a metric that measures the time difference between the expected shipment date and the actual date that the order is delivered to the customer. This helps identify and measure the efficiency and reliability of the shipping processes.

The calculation of Shipment Delay involves comparing the `Shipment Days - Actual` (actual date the order is shipped) with the `Shipment Days - Scheduled` (the expected or planned date of shipment).

#### Shipment Delay in Days
Shipment Delay  = [Shipment Days - Actual] - ['Shipment Days - Scheduled']

### Profit Margin

Profit Margin helps to assessing the profitability of the supply chain operations. It provides insights into the effectiveness of cost control and pricing strategies, enabling organizations to make adjustments to enhance overall profitability.

Profit Margin : Total Profit / Total Gross Sales *100

### Inventory to Sales Delta

This is metric which indicating how efficiently inventory is managed and whether there may be overstocking or understocking issues, allowing for more informed decisions to optimize inventory levels.

Inventory to Sales Delta = Total Warehouse Inventory - Total Order Quantity

### Under or Overstock

Under or Overstock helps to find which products are Overstock and which are understock.
And it helps to the balance of inventory.

Under or Overstock = Inventory to Sales Delta >0 THEN 'Overstock'
ELSE 'Understock'

# Data Exporting & Conclusion

In [29]:
# Final Check
df_orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 30871 entries, 0 to 30870
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Order ID                   30871 non-null  int64         
 1   Order Item ID              30871 non-null  int64         
 2   Order YearMonth            30871 non-null  int64         
 3   Order Quantity             30871 non-null  int64         
 4   Product Department         30871 non-null  str           
 5   Product Category           30871 non-null  str           
 6   Product Name               30871 non-null  str           
 7   Customer ID                30871 non-null  int64         
 8   Customer Market            30871 non-null  str           
 9   Customer Region            30871 non-null  str           
 10  Customer Country           30871 non-null  str           
 11  Warehouse Country          30871 non-null  str           
 12  Shipment Mode  

In [30]:
df_inventory.info()

<class 'pandas.DataFrame'>
RangeIndex: 4200 entries, 0 to 4199
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Product Name             4200 non-null   str    
 1   Year Month               4200 non-null   int64  
 2   Warehouse Inventory      4200 non-null   int64  
 3   Inventory Cost Per Unit  4200 non-null   float64
dtypes: float64(1), int64(2), str(1)
memory usage: 131.4 KB


In [31]:
df_fulfillment.info()

<class 'pandas.DataFrame'>
RangeIndex: 118 entries, 0 to 117
Data columns (total 2 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Product Name                        118 non-null    str    
 1   Warehouse Order Fulfillment (days)  118 non-null    float64
dtypes: float64(1), str(1)
memory usage: 2.0 KB


## Exporting Cleaned Data

Save the cleaned and processed DataFrames to a csv file for further visualization and analysis in tableau.

In [32]:
# Write the cleaned frames for Tableau to consume.
OUT = Path('Cleaned_Data')
OUT.mkdir(exist_ok=True)

df_orders.to_csv(OUT / 'orders_and_shipments_clean.csv', index=False)
df_inventory.to_csv(OUT / 'inventory_clean.csv', index=False)
df_fulfillment.to_csv(OUT / 'fulfillment_clean.csv', index=False)

for name, df in [('orders', df_orders), ('inventory', df_inventory),
                 ('fulfillment', df_fulfillment)]:
    print(f"{name:12s}: {df.shape[0]:>6,} rows x {df.shape[1]:>2} cols")

orders      : 30,871 rows x 21 cols
inventory   :  4,200 rows x  4 cols
fulfillment :    118 rows x  2 cols


## Conclusion

This notebook cleaned the `orders`, `inventory` and `fulfillment` tables and
resolved four issues that would otherwise have propagated into the dashboards:

1. **Missing discounts encoded as `'  -  '`** were converted to `NaN` rather than
   `0`, so aggregates exclude them instead of being pulled toward zero.
2. **A systematic off-by-one in processing time** - caused by subtracting a
   timestamped order date from an untimed shipment date - was removed. It had
   been understating processing time on 29,975 of 30,871 rows.
3. **Corrupt shipment dates** (~16% of rows, including shipments dated up to 975
   days *before* their order) are now flagged rather than silently averaged in.
4. **Duplicate detection** was moved onto business keys, since the original check
   ran on a unique identifier and could not have failed.

`Profit Margin`, `Inventory to Sales Delta`, `Storage Cost` and `Shipment Delay`
are calculated in Tableau from the exported files.

# Closing

As an aspiring data engineer/data analyst, I will consistently look for opportunities to improve my skills and insights.Thank you so much.

Throughout this project, I aimed to demonstrate data analytics expertise and provide actionable insights for real-world business challenges, using effective data processing, data cleaning, and advanced analytics techniques to generate meaningful conclusions and support decision making.